# Setting up a basic ReAct Agent in LangGraph

In [ ]:
import os

os.environ["OPENROUTER_API_KEY"] = 'sk-***'
os.environ["SERPAPI_API_KEY"] = 's***'  # https://serpapi.com for a free token!

In [2]:
from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["serpapi"])


In [3]:
tools[0].name

'Search'

In [4]:
tools[0].description

'A search engine. Useful for when you need to answer questions about current events. Input should be a search query.'

In [5]:
tools[0].run('Sinan Ozdemir')

'[\'Ozdemir Sinan : Sinan is an active lecturer focusing on large language models and a former lecturer of data science at the Johns Hopkins University. He is the author of multiple textbooks on data science and machine learning including "Quick Start Guide to LLMs". ...\', \'Sinan Ozdemir type: Author.\', \'Sinan Ozdemir entity_type: people.\', \'Sinan Ozdemir kgmid: /g/11hcjs9cr6.\', "I\'ve spent 12+ years at the intersection of AI research, education, and building: from founding Kylie.ai (YC, acquired) and patenting tool-use agents in 2018, ...", \'Helping companies leverage AI technology to solve complex problems. Founder, author, and consultant specializing in AI, LLMs, and data science.\', \'Data Scientist + Author + Entrepreneur. Check out my new book on LLMs on Amazon (Top 10 in AI/NLP) - sinanuozdemir.\', "A beginner\'s guide to essential math and coding skills for data fluency and machine learning by Sinan Ozdemir", \'Sinan is a former lecturer of Data Science at Johns Hopkin

# Use [OpenRouter](https://openrouter.ai/) for API inference

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from datetime import datetime
import os

openrouter_llm = ChatOpenAI(
    model="openai/gpt-4.1-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

today = datetime.now().strftime("%B %d, %Y")

agent_executor = create_agent(
    openrouter_llm, tools, system_prompt=f'Today is {today}')  # true as of today :)

In [7]:
response = agent_executor.invoke({"messages": [("user", "Who is the current Ravens QB?")]})


In [8]:
response['messages'][-1].content

'The current quarterback for the Baltimore Ravens is Lamar Jackson.'

In [10]:
import time
for llm in (
    'openai/gpt-oss-120b',
    'openai/gpt-4.1-mini',
    'openai/gpt-5-nano',
    'anthropic/claude-sonnet-4'
    ):  
    openrouter_llm = ChatOpenAI(
        model=llm,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        extra_body={'order': ['groq'], }
        
    )
    agent_executor = create_agent(openrouter_llm, tools, system_prompt=f'Today is {today}') 
    print(f'LLM: {llm}')
    start_time = time.time()
    print(agent_executor.invoke({"messages": [("user", "Who is the current Ravens QB?")]})['messages'][-1].content)
    end_time = time.time()
    print(f'Time taken: {end_time - start_time} seconds')

LLM: openai/gpt-oss-120b

Time taken: 0.4226710796356201 seconds
LLM: openai/gpt-4.1-mini
The current quarterback (QB) for the Baltimore Ravens is Lamar Jackson.
Time taken: 7.7173988819122314 seconds
LLM: openai/gpt-5-nano
Lamar Jackson is the Ravens’ starting quarterback in 2026. The backup options on the depth chart include Tyler Huntley, with others like Joe Fagnano and Diego Pavia also listed in the QB room. If you want, I can pull the latest depth chart or contract details.
Time taken: 12.98635220527649 seconds
LLM: anthropic/claude-sonnet-4
Based on the search results, **Lamar Jackson** is the current quarterback for the Baltimore Ravens. He wears #8 and is listed as the starting quarterback on their depth chart. The results show he's 29 years old, 6-2, 205 lbs, and is in his 9th year of experience as they prepare for the 2026 season. Tyler Huntley and Joe Fagnano are also listed as backup quarterbacks on the roster.
Time taken: 3.6280910968780518 seconds
